In [10]:
from google.colab import files
import os

# Membantu mengunggah file langsung jika belum ada
if not os.path.exists('Sales Transaction v.4a.csv'):
    print("Silakan unggah file 'Sales Transaction v.4a.csv':")
    uploaded = files.upload()
else:
    print("File 'Sales Transaction v.4a.csv' sudah terdeteksi di penyimpanan lokal.")

File 'Sales Transaction v.4a.csv' sudah terdeteksi di penyimpanan lokal.


# Pertemuan 5 — Membersihkan Data dengan Alasan
### Workbook Kelompok — dataset **Sales Transaction v.4a**

Notebook ini menghitung **semua angka langsung dari CSV penuh** saat dijalankan, lalu menulis dua laporan otomatis:
`data_quality_log` (kondisi data sebelum vs sesudah) dan `cleaning_log` (apa yang diubah, berapa baris, kenapa, alternatif yang ditolak).

> Jalankan dengan **Runtime → Run all**. Tidak ada angka yang diketik manual.

In [11]:
import pandas as pd, numpy as np, os
from google.colab import drive
drive.mount('/content/drive')

FOLDER = '/content/drive/MyDrive/SEMESTER-3/[2]EDA 2026/Pertemuan05'   # sesuaikan bila path berbeda
FILE   = os.path.join(FOLDER, 'Sales Transaction v.4a.csv')
if not os.path.exists(FILE):
    FILE = 'Sales Transaction v.4a.csv'           # fallback: file diunggah ke sesi Colab
OUT = FOLDER if os.path.isdir(FOLDER) else '.'
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 160)
print('Membaca:', FILE); print('Output  :', OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Membaca: Sales Transaction v.4a.csv
Output  : .


## Bagian 0 — Data mentah (tidak pernah diubah)
Semua kolom dibaca sebagai teks agar tidak ada konversi diam-diam oleh pandas.

In [12]:
mentah = pd.read_csv(FILE, dtype=str, keep_default_na=False, na_values=[''])
mentah.columns = [c.strip() for c in mentah.columns]
print('Data mentah:', mentah.shape)
mentah.head()

Data mentah: (536350, 8)


,TransactionNo,Date,ProductNo,ProductName,Price,Quantity,CustomerNo,Country
0,581482,12/9/2019,22485,Set Of 2 Wooden Market Crates,21.47,12,17490,United Kingdom
1,581475,12/9/2019,22596,Christmas Star Wish List Chalkboard,10.65,36,13069,United Kingdom
2,581475,12/9/2019,23235,Storage Tin Vintage Leaf,11.53,12,13069,United Kingdom
3,581475,12/9/2019,23272,Tree T-Light Holder Willie Winkie,10.65,12,13069,United Kingdom
4,581475,12/9/2019,23239,Set Of 4 Knick Knack Tins Poppies,11.94,6,13069,United Kingdom


## Bagian 1 — Data Quality Log: profil data
Fungsi `profil()` dijalankan **dua kali** (sebelum dan sesudah pembersihan) dengan definisi yang sama, supaya perbandingannya adil.

In [23]:
def profil(df):
    """Hitung metrik kualitas data. Dipakai untuk data mentah (str) maupun data bersih (bertipe)."""
    m = {}
    m["Jumlah baris"] = len(df)
    m["Jumlah kolom"] = df.shape[1]

    for c in [
        "TransactionNo",
        "Date",
        "ProductNo",
        "ProductName",
        "Price",
        "Quantity",
        "CustomerNo",
        "Country",
    ]:
        if c in df:
            m[f"Kosong: {c}"] = int(df[c].isna().sum())

    m["Baris duplikat persis"] = int(
        df[[c for c in mentah.columns]].duplicated().sum()
    )

    p = pd.to_numeric(df["Price"], errors="coerce")
    q = pd.to_numeric(df["Quantity"], errors="coerce")

    m["Price bukan angka (di luar kosong)"] = int(
        (p.isna() & df["Price"].notna()).sum()
    )
    m["Quantity bukan angka (di luar kosong)"] = int(
        (q.isna() & df["Quantity"].notna()).sum()
    )
    m["Price <= 0"] = int((p <= 0).sum())
    m["Quantity <= 0"] = int((q <= 0).sum())
    m["Transaksi pembatalan (awalan C)"] = int(
        df["TransactionNo"].astype(str).str.startswith("C").sum()
    )

    for c in ["TransactionNo", "ProductNo", "ProductName", "Country"]:
        s = df[c].dropna().astype(str)
        m[f"Spasi awal/akhir: {c}"] = int((s != s.str.strip()).sum())

    d = (
        df["Date"]
        if pd.api.types.is_datetime64_any_dtype(df["Date"])
        else pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")
    )
    m["Tanggal gagal dibaca"] = int(d.isna().sum())
    m["Jumlah negara unik"] = int(
        df["Country"].astype(str).str.strip().str.title().nunique()
    )
    m["ProductNo dengan >1 nama"] = int(
        (
            df.dropna(subset=["ProductName"])
            .groupby("ProductNo")["ProductName"]
            .nunique()
            > 1
        ).sum()
    )

    return pd.Series(m)


dq_sebelum = profil(mentah)
dq_sebelum.to_frame("Sebelum")

,Sebelum
Jumlah baris,536350
Jumlah kolom,8
Kosong: TransactionNo,0
Kosong: Date,0
Kosong: ProductNo,0
Kosong: ProductName,0
Kosong: Price,0
Kosong: Quantity,0
Kosong: CustomerNo,0
Kosong: Country,0


## Bagian 2 — Pembersihan dengan catatan
Aturan kelompok: **satu langkah, satu catatan**, dicatat saat itu juga. Setiap keputusan wajib punya *alasan* dan *alternatif yang ditolak*.

| Keputusan | Pilihan kami | Alternatif yang ditolak |
|---|---|---|
| Tanggal `12/9/2019` | dipilih otomatis dari uji dua urutan (lihat sel uji tanggal) | urutan lain — ditolak bila gagal-baca lebih banyak atau rentang tanggal tidak masuk akal |
| `CustomerNo` kosong | dibiarkan kosong + ditandai | diisi `0`/modus — ditolak karena mengarang pelanggan |
| Pembatalan (`C…`, qty negatif) | **ditandai** `is_batal`, tidak dibuang | dibuang — ditolak karena mengubah total penjualan tanpa jejak |
| `Price <= 0` / `Quantity <= 0` non-batal | dijadikan kosong + ditandai | dibuang — ditolak karena baris masih berguna untuk kolom lain |
| Duplikat persis | dibuang (simpan 1) | dibiarkan — ditolak karena menggandakan omzet; **risiko**: pembelian sah yang identik ikut terbuang |

In [14]:
d = mentah.copy()
jejak = []
def catat(langkah, terdampak, alasan, alternatif='-'):
    """Setiap perubahan dicatat SAAT dikerjakan."""
    jejak.append({'langkah': langkah, 'baris_terdampak': int(terdampak), 'alasan': alasan, 'alternatif_ditolak': alternatif})
    print(f'  {langkah}: {int(terdampak):,} baris')

# 1. Rapikan spasi pada kolom teks
n = 0
for c in ['TransactionNo','ProductNo','ProductName','Country']:
    s = d[c]; baru = s.str.strip().str.replace(r'\s+', ' ', regex=True)
    n += int((s.fillna('') != baru.fillna('')).sum()); d[c] = baru
catat('Rapikan spasi awal/akhir/ganda pada kolom teks', n,
      'Spasi tersembunyi memecah groupby dan nunique',
      'Dibiarkan - ditolak karena kategori sama terhitung berbeda')

  Rapikan spasi awal/akhir/ganda pada kolom teks: 0 baris


In [15]:
# 2. Price & Quantity -> angka (gagal dibaca dihitung, bukan dibiarkan senyap)
for c, nama in [('Price','Price'), ('Quantity','Quantity')]:
    sebelum_kosong = d[c].isna().sum()
    d[c] = pd.to_numeric(d[c], errors='coerce')
    catat(f'Konversi {nama} ke angka (gagal baca jadi kosong)', d[c].isna().sum() - sebelum_kosong,
          'Kolom harus numerik agar bisa dihitung; kegagalan dihitung agar kelihatan',
          'Berhenti di nilai aneh pertama - ditolak karena tidak tahu skala masalahnya')

# 3. CustomerNo -> bilangan bulat (kosong tetap kosong)
sebelum_kosong = d['CustomerNo'].isna().sum()
d['CustomerNo'] = pd.to_numeric(d['CustomerNo'], errors='coerce').astype('Int64')
catat('Konversi CustomerNo ke bilangan bulat', d['CustomerNo'].isna().sum() - sebelum_kosong,
      'ID pelanggan adalah pengenal, bukan hitungan; kekosongan dibiarkan agar jujur',
      'Isi 0 atau modus - ditolak karena mengarang pelanggan')
catat('CustomerNo kosong ditandai (tidak diisi)', d['CustomerNo'].isna().sum(),
      'Transaksi tanpa pelanggan tetap valid untuk analisis produk/negara',
      'Buang baris - ditolak karena menghilangkan penjualan sah')
d['tanpa_pelanggan'] = d['CustomerNo'].isna()

  Konversi Price ke angka (gagal baca jadi kosong): 0 baris
  Konversi Quantity ke angka (gagal baca jadi kosong): 0 baris
  Konversi CustomerNo ke bilangan bulat: 55 baris
  CustomerNo kosong ditandai (tidak diisi): 55 baris


In [16]:
# 4. Tanggal: uji dulu urutan bulan/hari, baru putuskan
t = d['Date']
bm = pd.to_datetime(t, format='%m/%d/%Y', errors='coerce')
bh = pd.to_datetime(t, format='%d/%m/%Y', errors='coerce')

print('Gagal baca  bulan/hari :', int(bm.isna().sum()), '| rentang', bm.min(), 's/d', bm.max())
print('Gagal baca  hari/bulan :', int(bh.isna().sum()), '| rentang', bh.min(), 's/d', bh.max())
print('Bagian pertama > 12 (pasti hari):', int((t.str.split('/').str[0].astype(float) > 12).sum()))
print('Bagian kedua  > 12 (pasti hari):', int((t.str.split('/').str[1].astype(float) > 12).sum()))

Gagal baca  bulan/hari : 0 | rentang 2018-12-01 00:00:00 s/d 2019-12-09 00:00:00
Gagal baca  hari/bulan : 305704 | rentang 2018-01-12 00:00:00 s/d 2019-12-10 00:00:00
Bagian pertama > 12 (pasti hari): 0
Bagian kedua  > 12 (pasti hari): 305704


In [17]:
# Keputusan diambil dari bukti di atas: format yang gagal-bacanya paling sedikit dan rentangnya masuk akal
pakai = '%m/%d/%Y' if bm.isna().sum() <= bh.isna().sum() else '%d/%m/%Y'
d['Date'] = bm if pakai == '%m/%d/%Y' else bh
catat(f'Konversi Date ke datetime (format {pakai})', d['Date'].isna().sum(),
      'Format ditentukan dari uji dua urutan, bukan ditebak; baris gagal baca dihitung',
      'Pakai format lain - ditolak karena lebih banyak gagal baca / rentang tidak masuk akal')

  Konversi Date ke datetime (format %m/%d/%Y): 0 baris


In [18]:
# 5. Pembatalan: ditandai, bukan dibuang
d['is_batal'] = d['TransactionNo'].astype(str).str.startswith('C') | (d['Quantity'] < 0)
catat('Tandai pembatalan/retur (awalan C atau Quantity negatif)', d['is_batal'].sum(),
      'Retur bukan kesalahan input; membuangnya mengubah omzet tanpa jejak',
      'Buang - ditolak; alternatif: pisah ke tabel retur pada tahap analisis')

# 6. Price/Quantity tidak masuk akal pada transaksi bukan-batal
bad_p = d['Price'] <= 0
bad_q = (d['Quantity'] <= 0) & ~d['is_batal']
d['anomali_nilai'] = bad_p | bad_q
d.loc[bad_p, 'Price'] = np.nan
d.loc[bad_q, 'Quantity'] = np.nan
catat('Price<=0 atau Quantity<=0 (bukan batal) dijadikan kosong + ditandai', d['anomali_nilai'].sum(),
      'Nilai mustahil; belum jelas penyebabnya, jadi tidak ditebak',
      'Buang baris - ditolak karena kolom lain masih berguna')

  Tandai pembatalan/retur (awalan C atau Quantity negatif): 8,585 baris
  Price<=0 atau Quantity<=0 (bukan batal) dijadikan kosong + ditandai: 0 baris


In [19]:
# 7. Duplikat persis -- hitung dulu, dibuang kemudian
kolom_asli = ['TransactionNo','Date','ProductNo','ProductName','Price','Quantity','CustomerNo','Country']
n_dup = int(d.duplicated(subset=kolom_asli).sum())
d = d.drop_duplicates(subset=kolom_asli, keep='first')
catat('Buang duplikat persis (simpan baris pertama)', n_dup,
      'Baris identik pada seluruh kolom menggandakan omzet',
      'Dibiarkan - ditolak; RISIKO: pembelian sah yang identik (mis. scan ulang) ikut terbuang')

# 8. Turunan untuk analisis
d['Revenue'] = d['Price'] * d['Quantity']
d['Bulan']   = d['Date'].dt.to_period('M').astype(str)
catat('Tambah kolom turunan Revenue dan Bulan', len(d),
      'Persiapan analisis; Revenue kosong bila Price/Quantity kosong (tidak dikarang)',
      '-')
d = d.reset_index(drop=True)
print('\nBaris mentah:', len(mentah), '->  baris bersih:', len(d))

  Buang duplikat persis (simpan baris pertama): 5,200 baris
  Tambah kolom turunan Revenue dan Bulan: 531,150 baris

Baris mentah: 536350 ->  baris bersih: 531150


## Bagian 3 — Data Quality Log: sebelum vs sesudah + verifikasi

In [20]:
# Kolom bantu tidak dihitung ulang; pakai profil yang sama pada data bersih
d_cek = d[mentah.columns].copy()
dq_sesudah = profil(d_cek)
dq = pd.DataFrame({'Sebelum': dq_sebelum, 'Sesudah': dq_sesudah})
dq['Selisih'] = dq['Sesudah'] - dq['Sebelum']
dq.index.name = 'Metrik'
dq

,Sebelum,Sesudah,Selisih
Metrik,,,
Jumlah baris,536350,531150,-5200
Jumlah kolom,8,8,0
Kosong: TransactionNo,0,0,0
Kosong: Date,0,0,0
Kosong: ProductNo,0,0,0
Kosong: ProductName,0,0,0
Kosong: Price,0,0,0
Kosong: Quantity,0,0,0
Kosong: CustomerNo,0,55,55


In [21]:
# Verifikasi mandiri: pastikan masalah yang diklaim hilang memang hilang
assert d.duplicated(subset=kolom_asli).sum() == 0, 'Masih ada duplikat'
assert (d['Price'].dropna() > 0).all(), 'Masih ada Price <= 0'
assert d['Date'].notna().all() or d['Date'].isna().sum() == int(dq_sesudah['Tanggal gagal dibaca']), 'Tanggal tidak konsisten'
print('Verifikasi lolos.')
print('Kolom yang BERTAMBAH kosong (bukan kegagalan, tanda keterbukaan):')
dq[(dq.index.str.startswith('Kosong')) & (dq['Selisih'] > 0)]

Verifikasi lolos.
Kolom yang BERTAMBAH kosong (bukan kegagalan, tanda keterbukaan):


,Sebelum,Sesudah,Selisih
Metrik,,,
Kosong: CustomerNo,0,55,55


## Bagian 4 — Tulis dua laporan ke Drive
Keluaran: `data_quality_log.md/.csv`, `cleaning_log.md/.csv`, dan data bersih. Angkanya langsung dari tabel di atas.

In [24]:
log = pd.DataFrame(jejak)
log.index = range(1, len(log) + 1)
log.index.name = "#"


def ke_md(df, judul, pembuka):
    baris = [f"# {judul}", "", pembuka, ""]
    kol = [df.index.name or ""] + list(df.columns)
    baris += ["| " + " | ".join(kol) + " |", "|" + "---|" * len(kol)]
    for idx, r in df.iterrows():
        baris.append("| " + " | ".join(str(x) for x in [idx, *r.values]) + " |")
    return "\n".join(baris) + "\n"


pembuka = f"Dataset: Sales Transaction v.4a.csv  \nBaris mentah: {len(mentah):,} -> baris bersih: {len(d):,}  \n"

open(
    os.path.join(OUT, "cleaning_log.md"), "w", encoding="utf-8"
).write(ke_md(log, "Cleaning Log", pembuka))

open(
    os.path.join(OUT, "data_quality_log.md"), "w", encoding="utf-8"
).write(ke_md(dq, "Data Quality Log", pembuka))

log.to_csv(os.path.join(OUT, "cleaning_log.csv"))
dq.to_csv(os.path.join(OUT, "data_quality_log.csv"))
d.to_csv(os.path.join(OUT, "sales_transaction_bersih.csv"), index=False)

print("Tersimpan di", OUT)
log

Tersimpan di .


,langkah,baris_terdampak,alasan,alternatif_ditolak
#,,,,
1,Rapikan spasi awal/akhir/ganda pada kolom teks,0,Spasi tersembunyi memecah groupby dan nunique,Dibiarkan - ditolak karena kategori sama terhi...
2,Konversi Price ke angka (gagal baca jadi kosong),0,Kolom harus numerik agar bisa dihitung; kegaga...,Berhenti di nilai aneh pertama - ditolak karen...
3,Konversi Quantity ke angka (gagal baca jadi ko...,0,Kolom harus numerik agar bisa dihitung; kegaga...,Berhenti di nilai aneh pertama - ditolak karen...
4,Konversi CustomerNo ke bilangan bulat,55,"ID pelanggan adalah pengenal, bukan hitungan; ...",Isi 0 atau modus - ditolak karena mengarang pe...
5,CustomerNo kosong ditandai (tidak diisi),55,Transaksi tanpa pelanggan tetap valid untuk an...,Buang baris - ditolak karena menghilangkan pen...
6,Konversi Date ke datetime (format %m/%d/%Y),0,"Format ditentukan dari uji dua urutan, bukan d...",Pakai format lain - ditolak karena lebih banya...
7,Tandai pembatalan/retur (awalan C atau Quantit...,8585,Retur bukan kesalahan input; membuangnya mengu...,Buang - ditolak; alternatif: pisah ke tabel re...
8,Price<=0 atau Quantity<=0 (bukan batal) dijadi...,0,"Nilai mustahil; belum jelas penyebabnya, jadi ...",Buang baris - ditolak karena kolom lain masih ...
9,Buang duplikat persis (simpan baris pertama),5200,Baris identik pada seluruh kolom menggandakan ...,Dibiarkan - ditolak; RISIKO: pembelian sah yan...


## Bagian 5 — Catatan untuk laporan
Yang perlu dibaca dari hasil di atas (isi angka setelah notebook dijalankan, jangan diketik dari ingatan):

1. **Pembersihan adalah rangkaian keputusan** — setiap baris `cleaning_log` punya alasan dan alternatif yang ditolak.
2. **Pembatalan tidak dibuang**, hanya ditandai; analisis penjualan harus memfilter `is_batal` secara sadar.
3. **Kosong yang bertambah bukan kegagalan** — itu penanda nilai mustahil/tidak terbaca yang tadinya tersamar.
4. **Risiko dedup** (pembelian identik yang sah ikut terbuang) wajib disebut di laporan sebagai keterbatasan.
5. Sel `cek(...)` pada template lama berisi kunci jawaban untuk dataset koperasi, jadi **tidak berlaku** untuk dataset Sales Transaction.